# 0.0 Setup

In [1]:
import os
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(PROJECT_DIR)

DATA_DIR = 'data'
DATA_PUBLIC_DIR = 'data/public'
DATA_PRIVATE_DIR = 'data/private'
DATA_PROCESSED_DIR = 'data/processed'
OUTPUT_DIR = 'output'

## 0.1. Modules

In [2]:
# We use the .py modules from the AAAI submission
from modules import public_coordinates_processor as pucp
from modules import private_coordinates_processor as prcp

# 1.0. Load datasets

## 1.1. Public Schools

In [3]:
%%time
# Initialize processor
processor_public_coordinates = pucp.PublicSchoolsProcessor(verbose=False)

# Process data with automatic coordinate fixing
processed_public_coordinates = processor_public_coordinates.process()

# Get quality report
quality_report = processor_public_coordinates.get_quality_report()
print("Data Quality Report:")
for key, value in quality_report.items():
    if isinstance(value, dict):
        print(f"  {key}: {len(value)} items")
    else:
        print(f"  {key}: {value}")

Data Quality Report:
  total_records: 47821
  missing_longitude: 632
  missing_latitude: 632
  missing_both: 632
  out_of_bounds: 73
  potentially_switched: 0
  valid_coordinates: 47116
  issues: []
  total_schools: 47821
  schools_with_valid_coords: 47116
  schools_with_missing_coords: 632
  coordinate_completeness_rate: 98.52575228456118
  regional_distribution: 17 items
  division_distribution: 10 items
CPU times: user 4.49 s, sys: 101 ms, total: 4.59 s
Wall time: 5.08 s


In [4]:
print(processed_public_coordinates.shape)
display(processed_public_coordinates.head())

(47821, 18)


,region,division,district,lis_school_id,nsbi_school_id,school_name,street_address,province,municipality,legislative_district,barangay,longitude,latitude,coord_valid,coord_missing,coord_out_of_bounds,coord_potentially_switched,school_id_processed
0,Region I,Ilocos Norte,Bacarra I,100001,100001.0,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,LIBTONG,120.614372,18.266860,True,False,False,False,100001
1,Region I,Ilocos Norte,Bacarra I,100002,100002.0,Bacarra CES,Santa Rita,ILOCOS NORTE,BACARRA,1st District,SANTA RITA (POB.),120.609487,18.251272,True,False,False,False,100002
2,Region I,Ilocos Norte,Bacarra I,100003,100003.0,Buyon ES,NONE,ILOCOS NORTE,BACARRA,1st District,BUYON,120.616050,18.234670,True,False,False,False,100003
3,Region I,Ilocos Norte,Bacarra I,100004,100004.0,Ganagan Elementary School,"#37 Ganagan,Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,GANAGAN,120.587415,18.250121,True,False,False,False,100004
4,Region I,Ilocos Norte,Bacarra I,100005,100005.0,Macupit ES,Macupit,ILOCOS NORTE,BACARRA,1st District,MACUPIT,120.641002,18.294093,True,False,False,False,100005


### Preprocess

In [5]:
# Sanity check if processed_school_id has any difference with lis_school_id
mask = processed_public_coordinates['lis_school_id'].astype(str) != processed_public_coordinates['school_id_processed'].astype(str)
processed_public_coordinates.loc[mask]

,region,division,district,lis_school_id,nsbi_school_id,school_name,street_address,province,municipality,legislative_district,barangay,longitude,latitude,coord_valid,coord_missing,coord_out_of_bounds,coord_potentially_switched,school_id_processed


In [6]:
# Check if the regions of this dataset follows 'old_region'
processed_public_coordinates['region'].unique()

array(['Region I', 'Region II', 'Region III', 'Region IV-A', 'MIMAROPA',
       'Region V', 'Region VI', 'Region VII', 'Region VIII', 'Region IX',
       'Region X', 'Region XI', 'Region XII', 'CARAGA', 'BARMM', 'CAR',
       'NCR'], dtype=object)

In [7]:
df_tmp = processed_public_coordinates.copy()

# Drop certain columns to reduce redundancy
drop_cols = ['nsbi_school_id','school_id_processed']
df_tmp.drop(columns=drop_cols, inplace=True)

# Rename certain columns for consistency with previously processed datasets
rename_cols = {
    'lis_school_id':'school_id',
    'region':'old_region',
}
df_tmp.rename(columns=rename_cols, inplace=True)

cols_str = df_tmp.loc[:,'old_region':'barangay'].columns
cols_float = df_tmp.loc[:, 'longitude':'latitude'].columns
cols_bool = df_tmp.loc[:, 'coord_valid':].columns
dtyps = [str, float, bool] 

for i, col_grp in enumerate([cols_str, cols_float, cols_bool]):
    dt = dtyps[i]
    for col in col_grp:
        df_tmp[col] = df_tmp[col].astype(dt)

# Distill dataframe to only include school IDs and essential columns
cols_ess = ['school_id','longitude','latitude','coord_valid','coord_missing','coord_out_of_bounds']
df_tmp = df_tmp[cols_ess]

In [8]:
print(df_tmp.shape)
display(df_tmp.head(3))

(47821, 6)


,school_id,longitude,latitude,coord_valid,coord_missing,coord_out_of_bounds
0,100001,120.614372,18.266860,True,False,False
1,100002,120.609487,18.251272,True,False,False
2,100003,120.616050,18.234670,True,False,False


### Save

In [35]:
%%time
fname = 'processed_public_school_coordinates.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_tmp.to_parquet(outpath)

CPU times: user 51.9 ms, sys: 3.92 ms, total: 55.8 ms
Wall time: 64.7 ms


## 1.2. Private Schools

In [4]:
%%time
# Fast processing with minimal logging
processor_private_coordinates = prcp.PrivateSchoolsProcessor(verbose=False)

# Process data efficiently
processed_data = processor_private_coordinates.process()

# Get summary
summary = processor_private_coordinates.get_summary()
print(f"Processed {summary['total_files_processed']} files")
print(f"Success rate: {summary['success_rate']:.1f}%")
# print(f"Final dataset: {summary['final_dataset_rows']} rows × {summary['final_dataset_columns']} columns")

Processed 16 files
Success rate: 100.0%
CPU times: user 1.4 s, sys: 155 ms, total: 1.55 s
Wall time: 2.86 s


In [5]:
# Validate coordinates if data was processed
if len(processed_data) > 0:
    # Perform coordinate validation
    coord_summary = processor_private_coordinates.validate_coordinates_with_reasons()

    processed_private_coordinates = processor_private_coordinates.replace_unclean_region_values()

In [6]:
display(processed_private_coordinates.head(3))

,region,division,district,beis_school_id,school_name,street_address,mother_school_id,province,municipality,legislative_district,barangay,sector,school_subclassification,modified_coc,latitude,longitude,excel_filename,sheet_name,coordinates_valid,coordinates_invalid_reason
0,CAR,Abra,Bangued East,400860,Data Center College of the Philippines,"Brgy. Ubbog, Lipcan, Bangued, Abra",NaN,ABRA,BANGUED (Capital),Lone District,LIPCAN,Private,Non-Sectarian,Purely SHS,17.58521,120.6149,CAR - List of Private School in the Philippin...,Abra,True,
1,CAR,Abra,Bangued East,406102,Abra Valley Colleges,McKinley Street Zone 4 Bangued Abra,NaN,ABRA,BANGUED (Capital),Lone District,ZONE 4 POB. (TOWN PROPER),Private,Non-Sectarian,All Offering,17.59657,120.6145,CAR - List of Private School in the Philippin...,Abra,True,
2,CAR,Abra,Bangued East,406104,Divine Word College-Bangued,Rizal,NaN,ABRA,BANGUED (Capital),Lone District,ZONE 6 POB. (SINAPANGAN),Private,Non-Sectarian,JHS with SHS,17.59447,120.61364,CAR - List of Private School in the Philippin...,Abra,True,


### Preprocess

In [7]:
processed_private_coordinates['region'].unique(),len(processed_private_coordinates['region'].unique())

(<StringArray>
 [        'CAR',      'CARAGA',    'MIMAROPA',         'NCR',    'Region I',
     'Region X',   'Region XI',  'Region XII',   'Region II',  'Region III',
  'Region IV-A',    'Region V',   'Region VI',  'Region VII', 'Region VIII',
    'Region IX']
 Length: 16, dtype: string,
 16)

In [8]:
df_tmp = processed_private_coordinates.copy()

# Drop certain columns to reduce redundancy
drop_cols = ['mother_school_id']
df_tmp.drop(columns=drop_cols, inplace=True)

# Rename certain columns for consistency with previously processed datasets
rename_cols = {
    'beis_school_id':'school_id',
    'region':'old_region',
    'coordinates_valid':'coord_valid'
}
df_tmp.rename(columns=rename_cols, inplace=True)

cols_str = df_tmp.loc[:,'old_region':'modified_coc'].columns.tolist() + ['excel_filename','sheet_name']
cols_float = df_tmp.loc[:, 'longitude':'latitude'].columns
cols_bool = df_tmp.loc[:, 'coord_valid':].columns
dtyps = [str, float, bool] 

for i, col_grp in enumerate([cols_str, cols_float, cols_bool]):
    dt = dtyps[i]
    for col in col_grp:
        df_tmp[col] = df_tmp[col].astype(dt)

# Distill dataframe to only include school IDs and essential columns
cols_ess = ['school_id','longitude','latitude','coord_valid']
df_tmp = df_tmp[cols_ess]

In [9]:
print(df_tmp.shape)
display(df_tmp.head(3))

(11831, 4)


,school_id,longitude,latitude,coord_valid
0,400860,120.6149,17.58521,True
1,406102,120.6145,17.59657,True
2,406104,120.61364,17.59447,True


### Save

In [10]:
%%time
fname = 'processed_private_school_coordinates.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_tmp.to_parquet(outpath)

CPU times: user 28 ms, sys: 19.4 ms, total: 47.4 ms
Wall time: 61.3 ms


### Load

In [4]:
%%time
fname = 'processed_private_school_coordinates.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_tmp = pd.read_parquet(outpath)
print(df_tmp.shape)

(43586, 6)
CPU times: user 62 ms, sys: 37.2 ms, total: 99.2 ms
Wall time: 99.5 ms
